# MNIST Unity Local PC Measurements

This notebook documents the local Windows PC measurement workflow used after the Unity MNIST application was functionally working.

The same Unity project and shared benchmark scripts were reused across CPU and GPU Compute testing. Moving between configurations primarily required:

- changing the Unity Inference Engine worker backend;
- activating the script associated with the test being run;
- assigning the required model and test-data assets;
- changing output labels so the recorded results identified the correct backend; and
- running the corresponding benchmark in either the Unity Editor or a Windows standalone build.

This notebook focuses on the **measurement procedure and configuration changes**. Project construction, package installation, Canvas setup, camera preprocessing, and Windows build creation are documented in the separate setup notebook.

## 1. Shared Unity Project

CPU and GPU measurements were not performed in separate Unity projects.

Instead, the same project was reused and the active inference backend was changed in the scripts.

The two worker configurations were:

### CPU

```csharp
worker = new Worker(runtimeModel, BackendType.CPU);
```

### GPU Compute

```csharp
worker = new Worker(runtimeModel, BackendType.GPUCompute);
```

The ONNX model, MNIST test binaries, scene layout, and most benchmark logic remained the same.

Some current scripts contain Quest-oriented labels or comments from later reuse of the same codebase. For PC reproduction, those labels should be changed to identify the Windows PC and selected backend. The measurement logic itself is the relevant part of the script.

## 2. Measurement Assets

The local PC measurements use:

```text
mnist_model.onnx
mnist_test_images.bin
mnist_test_labels.bin
```

The two MNIST test binaries are stored in the repository as:

```text
mnist_test_binaries.zip
```

The image binary contains the full 10,000-image MNIST test set in Unity-compatible float format. The label binary contains the expected digit for each test image.

The model is the same ONNX model documented in the separate MNIST model branches.

## 3. Measurement Scripts

The following scripts support the local PC measurements:

| Script | Measurement role |
|---|---|
| `MNISTAccuracyTest.cs` | Full 10,000-image classification accuracy |
| `MNISTLatencyTest.cs` | Controlled single-image latency benchmark |
| `MNISTThroughputTest.cs` | Repeated-inference throughput benchmark |
| `MNISTMemoryTest.cs` | Windows process-memory observations |
| `MNISTSustainedInference.cs` | Continuous inference workload |
| `CameraMNISTInference.cs` | Live camera classification and camera benchmarks |
| `MNISTSingleTest.cs` | Functional single-image validation |
| `MNISTModelLoader.cs` | Basic model/backend loading check |

Only the script needed for the current measurement should be active when practical.

The Unity Editor configuration was changed between tests rather than maintaining a separate scene for every benchmark.

## 4. CPU and GPU Reconfiguration

For each benchmark script, verify the worker declaration before running the test.

### CPU configuration

```csharp
worker = new Worker(runtimeModel, BackendType.CPU);
```

Update any identifying output strings to:

```text
Platform: Windows PC
Backend: CPU
```

### GPU Compute configuration

```csharp
worker = new Worker(runtimeModel, BackendType.GPUCompute);
```

Update any identifying output strings to:

```text
Platform: Windows PC
Backend: GPU Compute
```

Where the script generates CSV filenames, use the corresponding CPU or GPU identifier so the resulting files can be distinguished later.

## 5. Editor-Side Test Switching

The project was developed incrementally, so multiple benchmark scripts could exist in the same scene.

For each measurement:

1. open `FirstScene`;
2. select the object containing the benchmark components;
3. enable the script needed for the current test;
4. disable unrelated benchmark scripts when necessary;
5. assign `mnist_model.onnx`;
6. assign `mnist_test_images.bin`;
7. assign `mnist_test_labels.bin` when required;
8. verify the worker backend;
9. verify output labels; and
10. run the scene or create the standalone build.

The exact hierarchy changed as the project evolved, but the measurement procedure was based on activating the appropriate component rather than creating a separate project for every test.

## 6. Accuracy Validation

`MNISTAccuracyTest.cs` evaluates all 10,000 MNIST test images.

The script:

1. loads the ONNX model;
2. reads each 784-value image from `mnist_test_images.bin`;
3. schedules inference;
4. selects the largest of the ten outputs;
5. compares the prediction with `mnist_test_labels.bin`; and
6. reports the number correct and percentage accuracy.

The validated Unity result was:

```text
Correct: 9663 / 10000
Accuracy: 96.63%
```

The same trained model was used for CPU and GPU Compute testing, so the expected classification accuracy is the same when the model executes correctly.

## 7. Accuracy Evidence

The screenshot collection includes editor validation captures showing:

- successful single-image prediction; and
- full-test-set accuracy of 96.63%.

Representative files include:

```text
pc_cpu_accuracy_validation.png
pc_gpu_accuracy_validation.png
```

These screenshots document functional consistency between the imported ONNX model and the expected MNIST result.

## 8. Controlled Latency Benchmark

`MNISTLatencyTest.cs` measures inference latency using one MNIST test image.

The procedure is:

```text
50 warm-up runs
1000 measured runs
```

The measured section begins immediately before:

```csharp
worker.Schedule(inputTensor);
```

and ends after synchronous output readback:

```csharp
worker.PeekOutput().ReadbackAndClone()
```

This means the reported latency includes scheduling and synchronous result retrieval for the inference worker, but not the initial model-loading step.

## 9. Latency Output

The latency script records the individual measured runs and reports:

- mean;
- median;
- P95;
- minimum; and
- maximum.

Representative editor evidence includes:

```text
pc_cpu_latency_benchmark.png
pc_gpu_latency_benchmark.png
```

The script also writes a CSV file to Unity's persistent-data directory.

The CPU and GPU measurements should be kept separate because the only intended processing change is the selected Inference Engine backend.

## 10. Throughput Benchmark

`MNISTThroughputTest.cs` measures repeated inference throughput.

The controlled benchmark uses:

```text
50 warm-up runs
10000 measured inferences
```

The script cycles through the MNIST test images and calculates:

```text
elapsed time
images per second
average time per inference
```

Representative GPU evidence includes:

```text
pc_gpu_throughput_benchmark.png
```

The same procedure can be executed using the CPU backend by changing the worker configuration and output labels.

## 11. Sustained Inference

`MNISTSustainedInference.cs` runs inference continuously until the workload is manually stopped.

This script is useful when a longer-running workload is needed for:

- power observation;
- thermal observation;
- sustained resource use; or
- general runtime stability.

The script repeatedly cycles through the 10,000-image test set and reports the accumulated number of inferences and average processing rate when stopped.

For CPU testing, use:

```csharp
BackendType.CPU
```

For GPU testing, use:

```csharp
BackendType.GPUCompute
```

## 12. Windows Process-Memory Measurement

`MNISTMemoryTest.cs` measures Windows process memory using:

```csharp
Process.GetCurrentProcess()
```

The script records:

- baseline working set;
- model-loaded working set;
- peak working set;
- baseline private memory;
- model-loaded private memory;
- peak private memory;
- model increment; and
- peak increment.

The measurements are written to CSV.

The screenshot collection also contains Task Manager captures of `MNISTUnity.exe` during CPU and GPU runs.

## 13. CPU Memory Evidence

The CPU Task Manager captures are stored as:

```text
pc_cpu_memory_taskmanager_sample1.png
pc_cpu_memory_taskmanager_sample2.png
pc_cpu_memory_taskmanager_sample3.png
pc_cpu_memory_taskmanager_sample4.png
pc_cpu_memory_taskmanager_sample5.png
```

These screenshots provide external visual confirmation of the Windows process-memory behavior during local CPU execution.

Because the screenshots document different points during execution, they should be treated as observation samples rather than as interchangeable duplicates.

## 14. GPU Memory Evidence

The GPU configuration produced several groups of Task Manager evidence.

General GPU process-memory samples include files such as:

```text
pc_gpu_memory_taskmanager_sample1.png
pc_gpu_memory_taskmanager_sample2.png
```

Additional captures expose Windows Task Manager GPU-related columns, including:

- dedicated GPU memory;
- shared GPU memory;
- GPU engine;
- GPU utilization; and
- process working set.

These files provide supporting evidence for the GPU Compute execution configuration.

## 15. Editor Runs Versus Standalone Build Runs

Some tests were initially executed in the Unity Editor and then recreated in the Windows standalone build.

This produces intentional overlap in the screenshots.

The two execution contexts serve different purposes:

### Unity Editor

Useful for:

- script validation;
- Console output;
- inspecting scene state;
- confirming assigned assets; and
- debugging benchmark behavior.

### Windows standalone build

Useful for:

- observing the deployed application outside the Editor;
- reducing Editor-related interference;
- Task Manager process measurements;
- GPU-memory observations; and
- physical power measurements.

When two screenshots show the same benchmark, the notebook should identify whether the evidence came from the Editor or the standalone application rather than treating one as a duplicate automatically.

## 16. Live Camera Classification

`CameraMNISTInference.cs` combines:

```text
webcam capture
→ FPGA-matched ROI mapping
→ 28×28 sampling
→ grayscale conversion
→ binary threshold
→ tensor creation
→ inference
→ prediction
```

The application displays:

1. the live webcam image;
2. the current processed 28×28 image; and
3. the frozen 28×28 image used for classification.

The `Classify` button freezes the current processed image and performs local inference.

## 17. Frozen Camera Benchmark

The camera script includes a benchmark that freezes the current 28×28 input and repeatedly classifies that same tensor.

The procedure uses:

```text
50 warm-up runs
1000 measured runs
```

This isolates NN execution on a camera-derived input after preprocessing has already been completed.

Representative CPU evidence includes:

```text
pc_camera_frozen_benchmark_run1.png
pc_camera_frozen_benchmark_run2.png
```

Representative GPU build evidence includes:

```text
pc_build_gpu_camera_frozen_benchmark.png
```

## 18. End-to-End Camera Benchmark

The end-to-end camera benchmark includes:

- reading camera pixels;
- mapping the image to the FPGA-equivalent source geometry;
- 28×28 sampling;
- grayscale conversion;
- thresholding;
- tensor creation;
- inference; and
- synchronous output readback.

This is therefore a different measurement from the frozen-image benchmark.

Representative evidence includes:

```text
pc_camera_end_to_end_benchmark.png
pc_build_gpu_camera_end_to_end_benchmark.png
Unity_End_End_CPU(1).jpg
```

## 19. Live Camera Throughput

The live-throughput mode processes unique camera frames while the camera remains active.

The script counts completed predictions over elapsed real time.

Representative CPU evidence includes:

```text
pc_camera_live_throughput.png
Unity_CPU_Throughput(1).jpg
```

Representative GPU build evidence includes:

```text
pc_build_gpu_camera_live_throughput_run1.png
pc_build_gpu_camera_live_throughput_run2.png
```

The live-camera throughput test is application-level behavior and should not be interpreted as equivalent to the controlled 10,000-image throughput benchmark.

## 20. Camera Benchmark Evidence

The screenshot and photo collection contains both software captures and photographs of the running application.

Relevant examples include:

```text
pc_camera_preprocessing_preview.png
pc_camera_classification_result.png
pc_camera_frozen_benchmark_early_test.png
pc_camera_frozen_benchmark_run1.png
pc_camera_frozen_benchmark_run2.png
pc_camera_end_to_end_benchmark.png
pc_camera_live_throughput.png
Unity_CPU_Benchmark(1).jpg
Unity_End_End_CPU(1).jpg
Unity_Live_inference.jpg
Unity_Live_Inference_Overview.jpg
```

The photographs document the physical workstation and camera setup, while the screenshots preserve readable benchmark output.

## 21. Power Measurement Setup

PC power was measured externally using the **P4400 Kill A Watt** meter.

This gives a whole-system AC power boundary rather than process-level or component-level power.

The measurement therefore includes the PC system as supplied through the monitored outlet.

The meter's operation manual is stored separately in the Manuals and Specifications branch.

## 22. Idle Power

An idle-power observation was collected before sustained inference activity.

Supporting evidence includes:

```text
pc_idle_power_measurement.mp4
```

The idle observation provides the whole-system reference against which active inference power can be interpreted.

The exact numerical values should be transcribed from the measurement records or video rather than inferred from unrelated screenshots.

## 23. CPU Power

CPU inference power evidence includes:

```text
Unity_CPU_Power(1).jpg
pc_continuous_inference_power_measurement.mp4
```

The continuous-inference workload was used to maintain model activity while the external power meter was observed.

Because the Kill A Watt measures the whole PC, the recorded value represents system-level AC consumption during the workload rather than CPU package power alone.

## 24. GPU Power

GPU Compute power evidence includes:

```text
Unity_GPU_Power.jpg
pc_gpu_power_measurement_unity.mp4
```

As with the CPU measurement, this is whole-system AC power.

The relevant comparison is therefore:

```text
PC idle whole-system power
vs.
PC CPU-inference whole-system power
vs.
PC GPU-Compute whole-system power
```

rather than a direct CPU-chip versus GPU-chip power comparison.

## 25. Video Evidence

The local-PC data branch includes video documentation of several measurements and demonstrations:

```text
pc_gpu_power_measurement_unity.mp4
pc_live_camera_latency_measurement.mp4
pc_camera_end_to_end_latency_measurement.mp4
pc_idle_power_measurement.mp4
pc_live_unity_camera_demo.mp4
pc_unity_editor_offline_validation.mp4
pc_continuous_inference_power_measurement.mp4
pc_live_camera_throughput_measurement.mp4
```

These videos supplement the screenshots and CSV files by documenting the application and physical measurement process as it occurred.

## 26. Offline Validation

One video documents the Unity project operating without Internet access:

```text
pc_unity_editor_offline_validation.mp4
```

This is useful evidence that local PC inference did not require the cloud service.

The local configuration loads and executes the ONNX model through Unity Inference Engine on the PC itself.

## 27. Recommended Measurement Order

A practical reproduction sequence is:

```text
1. Verify model loading
2. Run single-image validation
3. Run 10,000-image accuracy
4. Run controlled latency
5. Run controlled throughput
6. Run memory test
7. Run sustained inference for power observation
8. Validate live camera classification
9. Run frozen-camera benchmark
10. Run end-to-end camera benchmark
11. Run live-camera throughput
12. Repeat relevant tests with the alternate backend
```

For each test, record whether it was run:

```text
CPU or GPU Compute
Editor or Windows standalone build
```

## 28. Reproducibility Notes

The Unity project evolved in place.

The source files preserved in the code branch therefore represent the shared implementation rather than immutable historical copies of every intermediate CPU and GPU script.

For reproduction:

- use the same script;
- select the required backend;
- update identifying output labels;
- activate the corresponding benchmark component;
- assign the required assets; and
- execute the measurement.

This reflects the actual development workflow more accurately than maintaining artificial duplicate source files for each backend.

## 29. What This Notebook Does Not Cover

This notebook does not document:

- creation of the ONNX model;
- FPGA inference;
- Meta Quest deployment;
- cloud-assisted inference;
- detailed statistical comparison across platforms; or
- interpretation of dissertation-wide performance/security trade-offs.

Those topics are documented in their corresponding model, implementation, platform, and dissertation analysis materials.

## 30. Summary of Local PC Measurement Categories

The local PC implementation produced evidence in five major categories:

```text
Model correctness
    → single-image and 10,000-image accuracy

Controlled inference performance
    → latency and throughput

Resource use
    → process memory and GPU-memory observations

Application-level camera performance
    → frozen-image latency
    → end-to-end latency
    → live throughput

Whole-system power
    → idle
    → CPU inference
    → GPU Compute inference
```

Together, these measurements characterize both the raw local inference behavior and the practical camera-based application behavior of the Unity PC implementation.